# Step-1 : Install the Libraries

In [1]:
#!pip install pandas 
#!pip install streamlit pyvis networkx pandas
#!pip install openpyxl

# Step 2 : Import libraries

In [2]:
import streamlit as st
import openpyxl
import pandas as pd
import networkx as nx
from pyvis.network import Network
import webbrowser
import os
import json

# Step 3: import data

In [3]:
# Importing the sample data from csv file
df = pd.read_excel("../input_file/snowflake_data_lineage.xlsx", sheet_name="data_lineage")

# Print dataframe (few rows)
df.head(10)
df.tail(10)
print('----------------------')

# Print the dataframe info
print('Info: \n',df.info())
print('----------------------')

# Print 
print('No. of rows & columns',df.shape)

----------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 743 entries, 0 to 742
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   child_object          743 non-null    object
 1   parent_object         743 non-null    object
 2   object_type           743 non-null    object
 3   extraction_timestamp  743 non-null    object
dtypes: object(4)
memory usage: 23.3+ KB
Info: 
 None
----------------------
No. of rows & columns (743, 4)


# Step - 4: Create Data visualization

In [ ]:

# ------------------ LOAD CORRECT SHEET ------------------
df = pd.read_excel("../input_file/snowflake_data_lineage.xlsx", sheet_name="data_lineage")

# ------------------ CLEAN COLUMN NAMES ------------------
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Columns:", df.columns.tolist())

# ------------------ SET COLUMN NAMES ------------------
parent_col = "parent_object"
child_col = "child_object"
type_col = "object_type"

# ------------------ VALIDATE ------------------
for col in [parent_col, child_col]:
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found. Available: {df.columns.tolist()}")

# ------------------ CLEAN DATA ------------------
df = df.dropna(subset=[parent_col, child_col])
df = df.drop_duplicates()

# ------------------ BUILD GRAPH ------------------
G = nx.DiGraph()

for _, row in df.iterrows():
    parent = str(row[parent_col]).strip()
    child = str(row[child_col]).strip()
    rel_type = str(row.get(type_col, "")).strip()

    if parent and child:
        G.add_edge(parent, child, label=rel_type)

print(f"Nodes: {len(G.nodes)}, Edges: {len(G.edges)}")

# ------------------ VISUALIZE ------------------
net = Network(height="750px", width="100%", directed=True)
net.barnes_hut()

# Add nodes
for node in G.nodes:
    net.add_node(node, label=node)

# Add edges
for source, target, data in G.edges(data=True):
    net.add_edge(source, target, label=data.get("label", ""), arrows="to")

# ------------------ SAVE TO output_file ------------------
output_dir = "../output_file"
output_path = os.path.join(output_dir, "lineage.html")

# Ensure folder exists (safe)
os.makedirs(output_dir, exist_ok=True)

net.write_html(output_path, notebook=False)

# ------------------ OPEN IN BROWSER ------------------
webbrowser.open(os.path.abspath(output_path))

Columns: ['child_object', 'parent_object', 'object_type', 'extraction_timestamp']
Nodes: 490, Edges: 743


True

In [1]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
import webbrowser
import os

# ------------------ LOAD DATA ------------------
df = pd.read_excel("../input_file/snowflake_data_lineage.xlsx", sheet_name="data_lineage")

# ------------------ CLEAN COLUMN NAMES ------------------
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Columns:", df.columns.tolist())

# ------------------ SET COLUMN NAMES ------------------
parent_col = "parent_object"
child_col = "child_object"
type_col = "object_type"

# ------------------ VALIDATE ------------------
for col in [parent_col, child_col]:
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found. Available: {df.columns.tolist()}")

# ------------------ CLEAN DATA ------------------
df = df.dropna(subset=[parent_col, child_col])
df = df.drop_duplicates()

# ------------------ BUILD GRAPH ------------------
G = nx.DiGraph()

for _, row in df.iterrows():
    parent = str(row[parent_col]).strip()
    child = str(row[child_col]).strip()
    rel_type = str(row.get(type_col, "")).strip()

    if parent and child:
        G.add_edge(parent, child, label=rel_type)

print(f"Nodes: {len(G.nodes)}, Edges: {len(G.edges)}")

# ------------------ VISUALIZE ------------------
net = Network(height="750px", width="100%", directed=True)
net.barnes_hut()

for node in G.nodes:
    net.add_node(node, label=node)

for source, target, data in G.edges(data=True):
    net.add_edge(source, target, label=data.get("label", ""), arrows="to")

# ------------------ SAVE HTML ------------------
output_dir = "../output_file"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "lineage.html")

net.write_html(output_path, notebook=False)

# ------------------ ADD FILTER UI ------------------
parents = df[parent_col].dropna().unique().tolist()
children = df[child_col].dropna().unique().tolist()

filter_html = f"""
<style>
#sidebar {{
    position: fixed;
    left: 0;
    top: 0;
    width: 260px;
    height: 100%;
    background: #f7f7f7;
    padding: 15px;
    border-right: 2px solid #ddd;
    overflow-y: auto;
    font-family: Arial;
}}

#mynetwork {{
    margin-left: 280px;
}}

select, button {{
    width: 100%;
    margin-top: 5px;
    padding: 6px;
}}

button {{
    margin-top: 15px;
    cursor: pointer;
    border: none;
}}

.apply-btn {{
    background-color: #4CAF50;
    color: white;
}}

.reset-btn {{
    background-color: #f44336;
    color: white;
}}
</style>

<div id="sidebar">
    <h3>Filters</h3>

    <label><b>Parent</b></label>
    <select id="parentFilter">
        <option value="">All</option>
        {''.join([f'<option value="{p}">{p}</option>' for p in parents[:200]])}
    </select>

    <label><b>Child</b></label>
    <select id="childFilter">
        <option value="">All</option>
        {''.join([f'<option value="{c}">{c}</option>' for c in children[:200]])}
    </select>

    <button class="apply-btn" onclick="applyFilter()">Apply Filters</button>
    <button class="reset-btn" onclick="resetGraph()">Reset</button>
</div>

<script>
var originalNodes = null;
var originalEdges = null;

function storeOriginal() {{
    if (!originalNodes) {{
        originalNodes = nodes.get();
        originalEdges = edges.get();
    }}
}}

function applyFilter() {{
    storeOriginal();

    var parent = document.getElementById("parentFilter").value;
    var child = document.getElementById("childFilter").value;

    var filteredNodesSet = new Set();
    var filteredEdges = [];

    originalEdges.forEach(function(edge) {{
        if ((parent === "" || edge.from == parent) &&
            (child === "" || edge.to == child)) {{

            filteredEdges.push(edge);
            filteredNodesSet.add(edge.from);
            filteredNodesSet.add(edge.to);
        }}
    }});

    var newNodes = originalNodes.filter(n => filteredNodesSet.has(n.id));

    nodes.clear();
    edges.clear();

    nodes.add(newNodes);
    edges.add(filteredEdges);
}}

function resetGraph() {{
    nodes.clear();
    edges.clear();

    nodes.add(originalNodes);
    edges.add(originalEdges);
}}

document.addEventListener("DOMContentLoaded", function() {{
    storeOriginal();
}});
</script>
"""

# ------------------ INJECT INTO HTML ------------------
with open(output_path, "r", encoding="utf-8") as f:
    html = f.read()

html = html.replace(
    '<div id="mynetwork"',
    filter_html + '\n<div id="mynetwork"'
)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(html)

# ------------------ OPEN IN BROWSER ------------------
webbrowser.open(os.path.abspath(output_path))

Columns: ['child_object', 'parent_object', 'object_type', 'extraction_timestamp']
Nodes: 490, Edges: 743


True

# dark good for now


In [21]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
import webbrowser
import os

# ------------------ LOAD DATA ------------------
df = pd.read_excel("../input_file/snowflake_data_lineage.xlsx", sheet_name="data_lineage")

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Columns:", df.columns.tolist())

parent_col = "parent_object"
child_col = "child_object"
type_col = "object_type"

for col in [parent_col, child_col]:
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found. Available: {df.columns.tolist()}")

df = df.dropna(subset=[parent_col, child_col]).drop_duplicates()

# ------------------ BUILD GRAPH ------------------
G = nx.DiGraph()

for _, row in df.iterrows():
    G.add_edge(
        str(row[parent_col]).strip(),
        str(row[child_col]).strip(),
        label=str(row.get(type_col, "")).strip()
    )

print(f"Nodes: {len(G.nodes)}, Edges: {len(G.edges)}")

# ------------------ VISUALIZE ------------------
net = Network(height="750px", width="100%", directed=True)
net.barnes_hut()

for node in G.nodes:
    net.add_node(node, label=node)

for source, target, data in G.edges(data=True):
    net.add_edge(source, target, label=data.get("label", ""), arrows="to")

# ------------------ SAVE HTML ------------------
output_dir = "../output_file"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "lineage.html")

net.write_html(output_path, notebook=False)

# ------------------ FILTER DATA ------------------
parents = df[parent_col].dropna().unique().tolist()
children = df[child_col].dropna().unique().tolist()

# ------------------ DARK UI ------------------
filter_html = f"""
<style>
body {{
    background-color: #0f172a;
    color: #ffffff;
}}

#sidebar {{
    position: fixed;
    left: 0;
    top: 0;
    width: 280px;
    height: 100%;
    background: #111827;
    padding: 20px;
    border-right: 1px solid #1f2937;
    overflow-y: auto;
    font-family: Arial;
}}

#mynetwork {{
    margin-left: 300px;
    background-color: #020617;
}}

h3 {{
    color: #38bdf8;
}}

label {{
    color: #ffffff;
    font-size: 14px;
}}

select {{
    width: 100%;
    padding: 8px;
    margin-top: 6px;
    background: #1e293b !important;
    color: #ffffff !important;
    border: 1px solid #334155;
    border-radius: 5px;
    appearance: none;
}}

select option {{
    background: #1e293b !important;
    color: #ffffff !important;
}}

button {{
    width: 100%;
    margin-top: 12px;
    padding: 8px;
    border: none;
    border-radius: 6px;
    cursor: pointer;
}}

.apply-btn {{
    background: #22c55e;
    color: white;
}}

.reset-btn {{
    background: #ef4444;
    color: white;
}}
</style>

<div id="sidebar">
    <h3>Lineage Filters</h3>

    <label>Parent</label>
    <select id="parentFilter">
        <option value="">All</option>
        {''.join([f'<option value="{p}">{p}</option>' for p in parents[:200]])}
    </select>

    <label>Child</label>
    <select id="childFilter">
        <option value="">All</option>
        {''.join([f'<option value="{c}">{c}</option>' for c in children[:200]])}
    </select>

    <label>Depth</label>
    <select id="depthFilter">
        <option value="1">1 Level</option>
        <option value="2">2 Levels</option>
        <option value="full">Full</option>
    </select>

    <button class="apply-btn" onclick="applyFilter()">Apply</button>
    <button class="reset-btn" onclick="resetGraph()">Reset</button>
</div>

<script>
var originalNodes = null;
var originalEdges = null;

function storeOriginal() {{
    if (!originalNodes) {{
        originalNodes = nodes.get();
        originalEdges = edges.get();
    }}
}}

function applyFilter() {{
    storeOriginal();

    var parent = document.getElementById("parentFilter").value;
    var child = document.getElementById("childFilter").value;

    let nodeSet = new Set();
    let filteredEdges = [];

    originalEdges.forEach(edge => {{
        if ((parent === "" || edge.from == parent) &&
            (child === "" || edge.to == child)) {{

            filteredEdges.push(edge);
            nodeSet.add(edge.from);
            nodeSet.add(edge.to);
        }}
    }});

    nodes.clear();
    edges.clear();

    nodes.add(originalNodes.filter(n => nodeSet.has(n.id)));
    edges.add(filteredEdges);
}}

function resetGraph() {{
    nodes.clear();
    edges.clear();
    nodes.add(originalNodes);
    edges.add(originalEdges);
}}

document.addEventListener("DOMContentLoaded", function() {{
    storeOriginal();
}});
</script>
"""

# ------------------ INJECT ------------------
with open(output_path, "r", encoding="utf-8") as f:
    html = f.read()

html = html.replace('<div id="mynetwork"', filter_html + '\n<div id="mynetwork"')

with open(output_path, "w", encoding="utf-8") as f:
    f.write(html)

# ------------------ OPEN ------------------
webbrowser.open(os.path.abspath(output_path))

Columns: ['child_object', 'parent_object', 'object_type', 'extraction_timestamp']
Nodes: 490, Edges: 743


True

# better

In [33]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
import webbrowser
import os

# ------------------ LOAD DATA ------------------
df = pd.read_excel("../input_file/snowflake_data_lineage.xlsx", sheet_name="data_lineage")

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

parent_col = "parent_object"
child_col = "child_object"
type_col = "object_type"

df = df.dropna(subset=[parent_col, child_col]).drop_duplicates()

# ------------------ BUILD GRAPH ------------------
G = nx.DiGraph()

for _, row in df.iterrows():
    G.add_edge(
        str(row[parent_col]).strip(),
        str(row[child_col]).strip(),
        label=str(row.get(type_col, "")).strip()
    )

# ------------------ VISUALIZE ------------------
net = Network(height="750px", width="100%", directed=True)
net.barnes_hut()

for node in G.nodes:
    net.add_node(node, label=node)

for source, target, data in G.edges(data=True):
    net.add_edge(source, target, label=data.get("label", ""), arrows="to")

# ------------------ SAVE HTML ------------------
output_dir = "../output_file"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "lineage.html")

net.write_html(output_path, notebook=False)

parents = df[parent_col].dropna().unique().tolist()
children = df[child_col].dropna().unique().tolist()

# ------------------ CUSTOM UI ------------------
filter_html = f"""
<style>
body {{
    background: #0f172a;
    color: white;
}}

#sidebar {{
    position: fixed;
    left: 0;
    top: 0;
    width: 280px;
    height: 100%;
    background: #111827;
    padding: 15px;
}}

#mynetwork {{
    margin-left: 300px;
}}

input {{
    width: 100%;
    padding: 6px;
    background: #1e293b;
    color: white;
    border: 1px solid #334155;
}}

.dropdown {{
    max-height: 150px;
    overflow-y: auto;
    background: #1e293b;
    border: 1px solid #334155;
}}

.dropdown div {{
    padding: 6px;
    color: white;
    cursor: pointer;
}}

.dropdown div:hover {{
    background: #334155;
}}

button {{
    width: 100%;
    margin-top: 10px;
    padding: 8px;
    border: none;
    border-radius: 5px;
}}

.apply-btn {{ background: #22c55e; }}
.reset-btn {{ background: #ef4444; }}
</style>

<div id="sidebar">
    <h3>Filters</h3>

    <label>Parent</label>
    <input type="text" id="parentInput" placeholder="Search parent...">
    <div id="parentDropdown" class="dropdown"></div>

    <label>Child</label>
    <input type="text" id="childInput" placeholder="Search child...">
    <div id="childDropdown" class="dropdown"></div>

    <button class="apply-btn" onclick="applyFilter()">Apply</button>
    <button class="reset-btn" onclick="resetGraph()">Reset</button>
</div>

<script>
var originalNodes = null;
var originalEdges = null;

var selectedParent = "";
var selectedChild = "";

const parents = {parents[:200]};
const children = {children[:200]};

function renderDropdown(list, container, setter) {{
    container.innerHTML = "";
    list.forEach(item => {{
        let div = document.createElement("div");
        div.innerText = item;
        div.onclick = () => {{
            setter(item);
        }};
        container.appendChild(div);
    }});
}}

document.getElementById("parentInput").oninput = function() {{
    let val = this.value.toLowerCase();
    renderDropdown(
        parents.filter(p => p.toLowerCase().includes(val)),
        document.getElementById("parentDropdown"),
        (v) => {{ selectedParent = v; this.value = v; }}
    );
}};

document.getElementById("childInput").oninput = function() {{
    let val = this.value.toLowerCase();
    renderDropdown(
        children.filter(c => c.toLowerCase().includes(val)),
        document.getElementById("childDropdown"),
        (v) => {{ selectedChild = v; this.value = v; }}
    );
}};

function storeOriginal() {{
    if (!originalNodes) {{
        originalNodes = nodes.get();
        originalEdges = edges.get();
    }}
}}

function applyFilter() {{
    storeOriginal();

    let nodeSet = new Set();
    let filteredEdges = [];

    originalEdges.forEach(edge => {{
        if ((selectedParent === "" || edge.from == selectedParent) &&
            (selectedChild === "" || edge.to == selectedChild)) {{

            filteredEdges.push(edge);
            nodeSet.add(edge.from);
            nodeSet.add(edge.to);
        }}
    }});

    nodes.clear();
    edges.clear();

    nodes.add(originalNodes.filter(n => nodeSet.has(n.id)));
    edges.add(filteredEdges);
}}

function resetGraph() {{
    nodes.clear();
    edges.clear();
    nodes.add(originalNodes);
    edges.add(originalEdges);
}}

document.addEventListener("DOMContentLoaded", function() {{
    storeOriginal();
}});
</script>
"""

# ------------------ INJECT ------------------
with open(output_path, "r", encoding="utf-8") as f:
    html = f.read()

html = html.replace('<div id="mynetwork"', filter_html + '\n<div id="mynetwork"')

with open(output_path, "w", encoding="utf-8") as f:
    f.write(html)

webbrowser.open(os.path.abspath(output_path))

True

# no clarity

In [1]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
import webbrowser
import os

# ------------------ LOAD DATA ------------------
df = pd.read_excel("../input_file/snowflake_data_lineage.xlsx", sheet_name="data_lineage")

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

parent_col = "parent_object"
child_col = "child_object"
type_col = "object_type"

df = df.dropna(subset=[parent_col, child_col]).drop_duplicates()

# ------------------ BUILD GRAPH ------------------
G = nx.DiGraph()

for _, row in df.iterrows():
    G.add_edge(
        str(row[parent_col]).strip(),
        str(row[child_col]).strip(),
        label=str(row.get(type_col, "")).strip()
    )

print(f"Nodes: {len(G.nodes)}, Edges: {len(G.edges)}")

# ------------------ VISUALIZE ------------------
net = Network(
    height="800px",
    width="100%",
    directed=True,
    bgcolor="#020617",
    font_color="white"
)

# ✅ SAFE physics config
net.set_options("""
{
  "physics": {
    "barnesHut": {
      "gravitationalConstant": -3000,
      "springLength": 220,
      "springConstant": 0.04
    }
  }
}
""")

# Add nodes
for node in G.nodes:
    net.add_node(node, label=node, color="#38bdf8")

# Add edges
for source, target, data in G.edges(data=True):
    net.add_edge(source, target, label=data.get("label", ""), arrows="to")

# ------------------ SAVE HTML ------------------
output_dir = "../output_file"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "lineage.html")

net.write_html(output_path, notebook=False)

# ------------------ PREPARE DATA ------------------
all_nodes = list(G.nodes)

# ------------------ UI HTML (NO f-string ❗) ------------------
custom_ui = """
<style>
body { background:#020617; color:white; font-family:Arial; }

#sidebar {
    position:fixed;
    left:0;
    top:0;
    width:300px;
    height:100%;
    background:#111827;
    padding:20px;
    border-right:1px solid #1f2937;
}

#mynetwork { margin-left:320px; }

input {
    width:100%;
    padding:8px;
    background:#1e293b;
    color:white;
    border:1px solid #334155;
    border-radius:5px;
}

.dropdown {
    max-height:150px;
    overflow-y:auto;
    background:#1e293b;
    margin-top:5px;
}

.dropdown div {
    padding:6px;
    cursor:pointer;
}

.dropdown div:hover {
    background:#334155;
}

button {
    width:100%;
    margin-top:12px;
    padding:10px;
    border:none;
    border-radius:6px;
    cursor:pointer;
}

.apply { background:#22c55e; }
.reset { background:#ef4444; }

h3 { color:#38bdf8; }
</style>

<div id="sidebar">
    <h3>Lineage Explorer</h3>

    <label>Search Node</label>
    <input id="searchBox" placeholder="Type name...">
    <div id="dropdown" class="dropdown"></div>

    <label>Mode</label>
    <select id="mode">
        <option value="both">Full Lineage</option>
        <option value="upstream">Upstream</option>
        <option value="downstream">Downstream</option>
    </select>

    <button class="apply" onclick="apply()">Apply</button>
    <button class="reset" onclick="reset()">Reset</button>
</div>

<script>

const allNodeNames = NODE_DATA;

let selectedNode = "";

const input = document.getElementById("searchBox");
const dropdown = document.getElementById("dropdown");

input.addEventListener("input", function() {
    const val = this.value.toLowerCase();
    dropdown.innerHTML = "";

    if (!val) return;

    allNodeNames
        .filter(n => n.toLowerCase().includes(val))
        .slice(0, 50)
        .forEach(n => {
            let div = document.createElement("div");
            div.innerText = n;

            div.onclick = () => {
                input.value = n;
                selectedNode = n;
                dropdown.innerHTML = "";
            };

            dropdown.appendChild(div);
        });
});

document.addEventListener("DOMContentLoaded", function () {

    const originalNodes = nodes.get();
    const originalEdges = edges.get();

    function getUpstream(start){
        let result = new Set([start]);
        let stack = [start];

        while(stack.length){
            let curr = stack.pop();
            originalEdges.forEach(e=>{
                if(e.to === curr && !result.has(e.from)){
                    result.add(e.from);
                    stack.push(e.from);
                }
            });
        }
        return result;
    }

    function getDownstream(start){
        let result = new Set([start]);
        let stack = [start];

        while(stack.length){
            let curr = stack.pop();
            originalEdges.forEach(e=>{
                if(e.from === curr && !result.has(e.to)){
                    result.add(e.to);
                    stack.push(e.to);
                }
            });
        }
        return result;
    }

    window.apply = function(){
        let node = selectedNode || input.value;
        let mode = document.getElementById("mode").value;

        if(!node) return;

        let selected = new Set();

        if(mode === "upstream") selected = getUpstream(node);
        else if(mode === "downstream") selected = getDownstream(node);
        else {
            selected = getUpstream(node);
            getDownstream(node).forEach(x=>selected.add(x));
        }

        let newNodes = originalNodes.filter(n => selected.has(n.id));
        let newEdges = originalEdges.filter(e => selected.has(e.from) && selected.has(e.to));

        nodes.clear();
        edges.clear();

        nodes.add(newNodes.map(n=>{
            if(n.id === node)
                return {...n, color:"#facc15", size:30};
            return n;
        }));

        edges.add(newEdges);
    }

    window.reset = function(){
        nodes.clear();
        edges.clear();
        nodes.add(originalNodes);
        edges.add(originalEdges);

        input.value = "";
        selectedNode = "";
    }

});
</script>
"""

# ------------------ INJECT DATA SAFELY ------------------
custom_ui = custom_ui.replace("NODE_DATA", str(all_nodes[:500]))

# ------------------ INJECT INTO HTML ------------------
with open(output_path, "r", encoding="utf-8") as f:
    html = f.read()

html = html.replace('<div id="mynetwork"', custom_ui + '\n<div id="mynetwork"')

with open(output_path, "w", encoding="utf-8") as f:
    f.write(html)

# ------------------ OPEN ------------------
webbrowser.open(os.path.abspath(output_path))

Nodes: 490, Edges: 743


True